# メインプレフィックス比較

対象は、**IPv4全体トラフィック**、**native** の選択済みプレフィックスmembership、およびRaw由来の固定プレフィックス集合です。**Raw** と **Broad** の条件で比較します。canonical `src_ip` と `dst_ip` は最初に観測した方向を保持しており、initiator/responderの役割を表しません。

In [2]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager

japanese_font_candidates = ('Hiragino Sans', 'Noto Sans CJK JP', 'Noto Serif CJK JP', 'IPAexGothic', 'YuGothic', 'Meiryo')
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
japanese_font = next((font for font in japanese_font_candidates if font in available_font_names), None)
if japanese_font is None:
    raise RuntimeError('日本語グリフを含むMatplotlibフォントが見つかりません')
plt.rcParams['font.family'] = japanese_font
plt.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

from mawi_global_analysis.comparison import build_comparison_flow_inclusion
from mawi_global_analysis.io import load_run

def resolve_analysis_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'mawi_global_analysis').is_dir():
            return candidate
    return Path.cwd()

root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', resolve_analysis_root())).resolve()
dataset_id = os.environ.get('MAWI_DATASET_ID', 'fixture')
run_name = os.environ.get('MAWI_RUN_NAME', 'baseline')
run = load_run(dataset_id, run_name, root=root)
condition_display_names = {'Raw': 'Raw（除外前）', 'Broad': 'Broad（除外後）'}
display_column_names = {'condition': '条件', 'flow_count': 'フロー数', 'packet_count': 'パケット数', 'frame_byte_count': 'フレームバイト数', 'ip_byte_count': 'IPバイト数', 'duration': '継続時間', 'median_packet_count': 'パケット数の中央値', 'median_frame_byte_count': 'フレームバイト数の中央値', 'median_duration': '継続時間の中央値', 'q90_duration': '継続時間の90パーセンタイル', 'q99_duration': '継続時間の99パーセンタイル', 'analysis_prefix': '分析プレフィックス', 'prefix': 'プレフィックス', 'prefix_length': 'プレフィックス長', 'seen_as_src_prefix': 'srcプレフィックスとして観測', 'seen_as_dst_prefix': 'dstプレフィックスとして観測', 'removed_flow_count': '除外フロー数', 'removed_flow_ratio': '除外フロー率', 'removed_packet_count': '除外パケット数', 'removed_packet_ratio': '除外パケット率', 'removed_frame_byte_count': '除外フレームバイト数', 'removed_frame_byte_ratio': '除外フレームバイト率', 'raw_selected_native_prefix_count': 'Raw選択済みnativeプレフィックス数', 'raw_table_prefix_count': 'Raw表のプレフィックス数', 'broad_table_prefix_count': 'Broad表のプレフィックス数', 'broad_zero_surviving_prefix_count': 'Broadで生存フローがゼロのプレフィックス数', 'raw_overall_flow_count': 'Raw全体フロー数', 'broad_overall_flow_count': 'Broad全体フロー数'}

def display_table(frame):
    rendered = frame.rename(columns=display_column_names).copy()
    if '条件' in rendered:
        rendered['条件'] = rendered['条件'].map(condition_display_names).fillna(rendered['条件'])
    display(rendered)

provenance = pd.DataFrame([{'データセット': dataset_id, '実行名': run_name, '分析対象': 'IPv4全体 / nativeプレフィックス / Raw-Broad', '設定ハッシュ': run.manifest.get('config', {}).get('hash'), '入力SHA-256': run.manifest.get('input', {}).get('sha256'), 'Gitコミット': run.manifest.get('git_commit')}])
display(provenance)

FileNotFoundError: run manifest not found: /Users/shunta/mawi-global-analysis/results/fixture/baseline/run_manifest.json

## 正規テーブルの導出

以下のすべての表示は、canonical `flows.csv`、M5で生成されたrun-localの `flow_labels.csv`、Raw由来の固定プレフィックスledger、および固定された `flow_prefix_membership.csv` からこのNotebook内で導出します。plot専用のpipelineデータは読みません。IPv4全体の対象を選ぶ前に、完全なcanonical flow/label集合へM6-A inclusion APIを適用します。

In [ ]:
# 完全なcanonical flow集合に対して判断を構築し、M6-A検証の前にflowを絞り込まない。
inclusion = build_comparison_flow_inclusion(run.flows, run.labels)
flow_metrics = run.flows.merge(inclusion, on='flow_id', how='inner', validate='one_to_one')
ipv4_flows = flow_metrics.loc[pd.to_numeric(flow_metrics['ip_version']) == 4]
raw_flows = ipv4_flows
selected_prefixes = run.prefixes.loc[run.prefixes['selected_for_analysis'] == True, ['prefix', 'prefix_length', 'seen_as_src_prefix', 'seen_as_dst_prefix']].copy()
native_membership = run.membership.loc[run.membership['analysis_scope'] == 'native'].copy()
selected_prefix_ids = set(selected_prefixes['prefix'])
if not set(native_membership['flow_id']).issubset(set(inclusion['flow_id'])):
    raise ValueError('native membershipにcanonical flow集合の外部のflow_idが含まれています')
if not set(native_membership['analysis_prefix']).issubset(selected_prefix_ids):
    raise ValueError('native membershipに固定Raw由来選択集合の外部のprefixが含まれています')
prefix_flow_columns = ['flow_id', 'packet_count', 'frame_byte_count', 'ip_byte_count', 'duration', 'raw_included', 'strict_included', 'broad_included']
prefix_flows = native_membership.merge(ipv4_flows.loc[:, prefix_flow_columns], on='flow_id', how='inner', validate='many_to_one')
selected_scope_flows = raw_flows.loc[raw_flows['flow_id'].isin(native_membership['flow_id'].unique())]
display(pd.DataFrame([{'IPv4 Rawフロー数': len(raw_flows), '選択済みnativeプレフィックス数': len(selected_prefixes), 'native membership行数': len(native_membership), 'いずれかのnativeプレフィックスに一致する一意なフロー数': native_membership['flow_id'].nunique()}]))
display_table(selected_prefixes.sort_values('prefix').head(20))

## Raw / Broad 比較表

これらは比較用の表です。すべての条件で同じRaw由来nativeプレフィックス集合とmembershipを保持します。比較可視化はM6-B2で行います。

In [ ]:
conditions = pd.DataFrame({'condition': ['Raw', 'Broad'], 'inclusion_column': ['raw_included', 'broad_included']})
count_columns = ['flow_count', 'packet_count', 'frame_byte_count']

def condition_metrics(frame, condition, inclusion_column):
    survivors = frame.loc[frame[inclusion_column]]
    return {'condition': condition, 'flow_count': len(survivors), 'packet_count': survivors['packet_count'].sum(), 'frame_byte_count': survivors['frame_byte_count'].sum(), 'median_packet_count': survivors['packet_count'].median(), 'median_frame_byte_count': survivors['frame_byte_count'].median(), 'median_duration': survivors['duration'].median()}

def safe_ratio(numerator, denominator):
    return numerator.div(denominator.where(denominator > 0))

overall_condition_summary = pd.DataFrame([condition_metrics(ipv4_flows, row.condition, row.inclusion_column) for row in conditions.itertuples(index=False)])
raw_overall = overall_condition_summary.loc[overall_condition_summary['condition'] == 'Raw'].iloc[0]
overall_removal_rows = []
for row in overall_condition_summary.loc[overall_condition_summary['condition'] == 'Broad'].itertuples(index=False):
    overall_removal_rows.append({'condition': row.condition, 'removed_flow_count': raw_overall.flow_count - row.flow_count, 'removed_flow_ratio': (raw_overall.flow_count - row.flow_count) / raw_overall.flow_count if raw_overall.flow_count else np.nan, 'removed_packet_count': raw_overall.packet_count - row.packet_count, 'removed_packet_ratio': (raw_overall.packet_count - row.packet_count) / raw_overall.packet_count if raw_overall.packet_count else np.nan, 'removed_frame_byte_count': raw_overall.frame_byte_count - row.frame_byte_count, 'removed_frame_byte_ratio': (raw_overall.frame_byte_count - row.frame_byte_count) / raw_overall.frame_byte_count if raw_overall.frame_byte_count else np.nan})
overall_removal_summary = pd.DataFrame(overall_removal_rows)

prefix_condition_index = selected_prefixes.loc[:, ['prefix']].rename(columns={'prefix': 'analysis_prefix'}).merge(conditions.loc[:, ['condition', 'inclusion_column']], how='cross')
prefix_condition_metrics = []
for row in conditions.itertuples(index=False):
    survivors = prefix_flows.loc[prefix_flows[row.inclusion_column]]
    metrics = survivors.groupby('analysis_prefix', as_index=False).agg(flow_count=('flow_id', 'size'), packet_count=('packet_count', 'sum'), frame_byte_count=('frame_byte_count', 'sum'), median_packet_count=('packet_count', 'median'), median_frame_byte_count=('frame_byte_count', 'median'), median_duration=('duration', 'median'))
    metrics['condition'] = row.condition
    prefix_condition_metrics.append(metrics)
per_prefix_condition_summary = prefix_condition_index.drop(columns='inclusion_column').merge(pd.concat(prefix_condition_metrics, ignore_index=True), on=['analysis_prefix', 'condition'], how='left', validate='one_to_one')
per_prefix_condition_summary.loc[:, count_columns] = per_prefix_condition_summary.loc[:, count_columns].fillna(0)

raw_prefix_metrics = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Raw', ['analysis_prefix', *count_columns]].rename(columns={column: f'raw_{column}' for column in count_columns})
per_prefix_removal_rows = []
for condition in ['Broad']:
    survivors = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, ['analysis_prefix', *count_columns]]
    comparison = raw_prefix_metrics.merge(survivors, on='analysis_prefix', how='left', validate='one_to_one')
    per_prefix_removal_rows.append(pd.DataFrame({'analysis_prefix': comparison['analysis_prefix'], 'condition': condition, 'removed_flow_count': comparison['raw_flow_count'] - comparison['flow_count'], 'removed_flow_ratio': safe_ratio(comparison['raw_flow_count'] - comparison['flow_count'], comparison['raw_flow_count']), 'removed_packet_count': comparison['raw_packet_count'] - comparison['packet_count'], 'removed_packet_ratio': safe_ratio(comparison['raw_packet_count'] - comparison['packet_count'], comparison['raw_packet_count']), 'removed_frame_byte_count': comparison['raw_frame_byte_count'] - comparison['frame_byte_count'], 'removed_frame_byte_ratio': safe_ratio(comparison['raw_frame_byte_count'] - comparison['frame_byte_count'], comparison['raw_frame_byte_count'])}))
per_prefix_removal_summary = pd.concat(per_prefix_removal_rows, ignore_index=True)

strict_overall_count = int(ipv4_flows['strict_included'].sum())
broad_overall_count = int(ipv4_flows['broad_included'].sum())
raw_overall_count = int(ipv4_flows['raw_included'].sum())
if not (broad_overall_count <= strict_overall_count <= raw_overall_count):
    raise ValueError('内部除外不変条件 Broad <= Strict <= Raw に違反しています')
overall_counts = overall_condition_summary.set_index('condition')['flow_count']
prefix_counts_by_condition = per_prefix_condition_summary.groupby('condition')['analysis_prefix'].nunique().reindex(conditions['condition'])
if not (overall_counts['Broad'] <= overall_counts['Raw']):
    raise ValueError('全体の生存フロー数が Broad <= Raw に違反しています')
if not (prefix_counts_by_condition == len(selected_prefixes)).all():
    raise ValueError('比較条件が完全な固定Raw由来プレフィックス集合を保持していません')
for condition in conditions['condition']:
    condition_prefixes = set(per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, 'analysis_prefix'])
    if condition_prefixes != selected_prefix_ids:
        raise ValueError('比較条件が固定Raw由来プレフィックス集合を変更しています')
sanity_summary = pd.DataFrame([{'raw_selected_native_prefix_count': len(selected_prefixes), 'raw_table_prefix_count': prefix_counts_by_condition['Raw'], 'broad_table_prefix_count': prefix_counts_by_condition['Broad'], 'broad_zero_surviving_prefix_count': int((per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Broad', 'flow_count'] == 0).sum()), 'raw_overall_flow_count': overall_counts['Raw'], 'broad_overall_flow_count': overall_counts['Broad']}])

display_table(overall_condition_summary)
display_table(overall_removal_summary)
display_table(per_prefix_condition_summary.sort_values(['condition', 'analysis_prefix']))
display_table(per_prefix_removal_summary.sort_values(['condition', 'analysis_prefix']))
display_table(sanity_summary)

## Raw / Broad 比較の可視化

これらの図は、作成済みの比較表における変化を示します。flowの再分類や固定Raw由来プレフィックス集合の変更は行いません。

In [ ]:
condition_order = ['Raw', 'Broad']
condition_labels = condition_display_names
median_metrics = ['median_packet_count', 'median_frame_byte_count', 'median_duration']
metric_labels = {'median_packet_count': 'フローあたりパケット数の中央値', 'median_frame_byte_count': 'フローあたりフレームバイト数の中央値', 'median_duration': '継続時間の中央値（秒）'}
overall_medians = overall_condition_summary.set_index('condition').reindex(condition_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, metric in zip(axes, median_metrics):
    axis.plot(condition_order, overall_medians[metric], marker='o', linewidth=2, color='tab:blue')
    axis.set(title=f'IPv4全体: {metric_labels[metric]}', xlabel='条件', ylabel=metric_labels[metric])
fig.suptitle('除外条件による全体中央値の変化', y=1.02)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
jitter = np.random.default_rng(0)
for axis, metric in zip(axes, median_metrics):
    for position, condition in enumerate(condition_order):
        values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, metric].dropna()
        axis.scatter(position + jitter.uniform(-0.12, 0.12, len(values)), values, s=16, alpha=0.45, color='tab:orange', label='選択済みnativeプレフィックス' if position == 0 else None)
        if not values.empty:
            axis.scatter(position, values.median(), marker='_', s=450, linewidths=2.5, color='tab:orange', zorder=2, label='プレフィックス中央値' if position == 0 else None)
        axis.scatter(position, overall_medians.loc[condition, metric], marker='D', s=70, color='tab:blue', zorder=3, label='IPv4全体' if position == 0 else None)
    axis.set(title=metric_labels[metric], xlabel='条件', ylabel=metric_labels[metric], xticks=range(len(condition_order)), xticklabels=[condition_labels[condition] for condition in condition_order])
    axis.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:orange', markersize=7, label='選択済みnativeプレフィックス'), Line2D([0], [0], marker='_', color='tab:orange', markersize=13, label='プレフィックス中央値'), Line2D([0], [0], marker='D', color='w', markerfacecolor='tab:blue', markersize=7, label='IPv4全体')])
fig.suptitle('IPv4全体と選択済みnativeプレフィックス中央値分布の比較', y=1.02)
fig.tight_layout()

### Rawからのプレフィックス別中央値の変化

各点は固定された1つの `analysis_prefix` を表します。恒等線に近いほど、そのプレフィックスの中央値変化が小さいことを示します。生存flowがない行は比較表に残しますが、中央値は `NaN` となるため図には表示しません。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(5.5, 14))
for row_index, metric in enumerate(median_metrics):
    raw_values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Raw', ['analysis_prefix', metric]].rename(columns={metric: 'raw_value'})
    condition = 'Broad'
    axis = axes[row_index]
    compared_values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, ['analysis_prefix', metric]].rename(columns={metric: 'condition_value'})
    paired = raw_values.merge(compared_values, on='analysis_prefix', how='inner', validate='one_to_one').dropna()
    if paired.empty:
        axis.text(0.5, 0.5, '対応する生存プレフィックス中央値はありません', ha='center', va='center', transform=axis.transAxes)
        axis.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
        axis.set(xlim=(0, 1), ylim=(0, 1))
    else:
        upper = max(paired['raw_value'].max(), paired['condition_value'].max(), 1)
        axis.scatter(paired['raw_value'], paired['condition_value'], s=18, alpha=0.6, color='tab:purple')
        axis.plot([0, upper], [0, upper], linestyle='--', color='black', linewidth=1)
        axis.set(xlim=(0, upper), ylim=(0, upper))
    axis.set(title=f'Rawと{condition_labels[condition]}の比較', xlabel=f'Raw {metric_labels[metric]}', ylabel=f'{condition_labels[condition]} {metric_labels[metric]}')
fig.suptitle('Rawからのプレフィックス別中央値の変化（恒等線）', y=0.995)
fig.tight_layout()

In [ ]:
removal_metrics = ['removed_flow_ratio', 'removed_packet_ratio', 'removed_frame_byte_ratio']
removal_labels = ['フロー', 'パケット', 'フレームバイト']
removal_plot = overall_removal_summary.set_index('condition').reindex(['Broad'])
positions = np.arange(len(removal_metrics))
width = 0.55
fig, axis = plt.subplots(figsize=(8, 4.5))
condition = 'Broad'
axis.bar(positions, removal_plot.loc[condition, removal_metrics], width=width, label=condition_labels[condition])
axis.set(title='全体IPv4トラフィックに対する除外量', xlabel='対象', ylabel='Rawに対する除外率', xticks=positions, xticklabels=removal_labels, ylim=(0, 1))
axis.legend()
fig.tight_layout()

## Rawベースラインの記述的分布

これらのRawのみのフロー長表示は、ベースラインの記述的な参照として残します。Raw/Broadの分布比較はこの節の対象外です。

In [ ]:
def ecdf(values):
    ordered = np.sort(np.asarray(values, dtype=float))
    return ordered, np.arange(1, len(ordered) + 1) / len(ordered)

protocols = {'TCP': ['6', '6.0', 'tcp'], 'UDP': ['17', '17.0', 'udp']}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for protocol, values in protocols.items():
    for scope, frame, style in [('IPv4全体', raw_flows, '-'), ('nativeプレフィックス関連', selected_scope_flows, '--')]:
        packets = frame.loc[frame['protocol'].astype(str).isin(values), 'packet_count']
        if packets.empty:
            continue
        x, y = ecdf(packets)
        label = f'{protocol}: {scope}'
        axes[0].hist(packets, bins='auto', histtype='step', linewidth=2, linestyle=style, label=label)
        axes[1].step(x, y, where='post', linestyle=style, label=label)
axes[0].set(title='パケット数ヒストグラム', xlabel='フローあたりパケット数', ylabel='フロー数', xscale='log', yscale='log')
axes[1].set(title='パケット数のECDF', xlabel='フローあたりパケット数', ylabel='累積確率', xscale='log')
for axis in axes:
    axis.legend()
fig.tight_layout()

In [ ]:
# canonical membershipがゼロのプレフィックスを含め、選択済みnativeプレフィックスごとに1行を保持する。
prefix_metrics = prefix_flows.groupby('analysis_prefix', as_index=False).agg(flow_count=('flow_id', 'size'), packet_count=('packet_count', 'sum'), frame_byte_count=('frame_byte_count', 'sum'), ip_byte_count=('ip_byte_count', 'sum'), median_packet_count=('packet_count', 'median'), median_frame_byte_count=('frame_byte_count', 'median'), median_duration=('duration', 'median'), q90_duration=('duration', lambda values: values.quantile(0.9)), q99_duration=('duration', lambda values: values.quantile(0.99)))
prefix_summary = selected_prefixes.rename(columns={'prefix': 'analysis_prefix'}).merge(prefix_metrics, on='analysis_prefix', how='left', validate='one_to_one')
zero_volume_prefixes = prefix_summary['flow_count'].isna()
prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']] = prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']].fillna(0)
display(pd.DataFrame([{'canonical membershipがゼロの選択済みプレフィックス数': int(zero_volume_prefixes.sum()), '選択済みプレフィックス数': len(prefix_summary)}]))
display_table(prefix_summary.sort_values('frame_byte_count', ascending=False).head(20))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
positive_volume = prefix_summary.loc[prefix_summary['frame_byte_count'] > 0]
for axis, y_column, title in zip(axes, ['median_packet_count', 'median_frame_byte_count', 'median_duration'], ['トラフィック量とパケット数中央値', 'トラフィック量とフレームバイト数中央値', 'トラフィック量と継続時間中央値']):
    axis.scatter(positive_volume['frame_byte_count'], positive_volume[y_column], alpha=0.75)
    axis.set(xscale='log', yscale='log' if (positive_volume[y_column] > 0).all() else 'linear', xlabel='nativeプレフィックスのフレームバイト数', ylabel=display_column_names[y_column], title=title)
fig.tight_layout()

In [ ]:
# この統合スコープ表示では、選択済みプレフィックス系列をflow_idで重複除去する。
fig, axis = plt.subplots(figsize=(6, 4.5))
for label, values in [('IPv4全体（Raw）', raw_flows['duration']), ('いずれかの選択済みnativeプレフィックス（Raw）', selected_scope_flows['duration'])]:
    x, y = ecdf(values)
    axis.step(x, y, where='post', label=label)
axis.set(title='フロー継続時間のECDF', xlabel='継続時間（秒）', ylabel='累積確率')
axis.legend()
fig.tight_layout()

この修正済みベースラインでは、選択された重複しないすべてのnative IPv4プレフィックスを意図的に使用します。prefix membershipは `src_ip ∈ prefix OR dst_ip ∈ prefix` とし、`src_match` と `dst_match` はトラフィックの役割ではなく観測方向の事実として保持します。